In [1]:
import os
import json
import re
import pandas as pd

pd.options.display.max_columns = 10

In [2]:
def extract_json(response: str):
    """Extract JSON content from a formatted string."""
    match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if match:
        json_str = match.group(1)
    else:
        json_str = response.strip('```json').strip('```')
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Chyba při dekódování JSON: {e}")
        return None

In [3]:
def process_txt_files(folder_path, prefix):
    """Zpracuje všechny txt soubory začínající prefixem (např. 'bank_part') ve složce a vrátí Pandas DataFrame."""
    all_data = []
    
    # Get files with prefix and end with .txt
    files = [f for f in os.listdir(folder_path) if f.startswith(prefix) and f.endswith(".txt")]
    
    # Order by number
    files.sort(key=lambda x: int(re.search(r'chunk(\d+)', x).group(1)))
    
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as file:
            for line in file:
                try:
                    json_obj = json.loads(line.strip())
                    content_str = json_obj.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
                    extracted_json = extract_json(content_str)
                    
                    if extracted_json:
                        row = {"id": json_obj["id"], "custom_id": json_obj["custom_id"]}
                        for feature in extracted_json.get("features", []):
                            row[feature["feature_name"]] = feature["answer"]
                        
                        all_data.append(row)
                except json.JSONDecodeError:
                    print(f"Chyba dekódování JSON v souboru {filename}")

    df = pd.DataFrame(all_data)
    return df

In [4]:
# Použití skriptu
folder_path = "../../data/outputs/bank77"
df = process_txt_files(folder_path, "bank77")

# Zobrazení výsledného dataframe
df

,id,custom_id,intent_category,query_length,contains_question,...,contains_refund_mention,contains_exchange_mention,contains_account_mention,contains_withdrawal_mention,contains_deposit_mention
0,batch_req_67c78c3682248190b4547edea94cf401,0,activate_my_card,short,yes,...,no,no,no,no,no
1,batch_req_67c78c36972c8190be4029aacf915797,1,activate_my_card,medium,yes,...,no,no,no,no,no
2,batch_req_67c78c36b3448190b0f6238033c438c7,2,activate_my_card,medium,yes,...,no,no,no,no,no
3,batch_req_67c78c36cad881909e7b8d5c15fa0291,3,track_my_card,medium,yes,...,no,no,no,no,no
4,batch_req_67c78c36dfe08190b02d235285506a1f,4,card_status,medium,yes,...,no,no,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...
13078,batch_req_67cb1a70277481909676fa211d09699c,3075,activate_my_card,short,yes,...,no,no,no,no,no
13079,batch_req_67cb1a703a848190820c3143b67f239c,3076,support_query,short,yes,...,no,no,no,no,no
13080,batch_req_67cb1a704ae88190a4bd560fddeec8ac,3077,business_inquiry,short,yes,...,no,no,no,no,no
13081,batch_req_67cb1a705b5c8190b7bd634d097bb6a9,3078,unknown,short,yes,...,no,no,no,no,no


# Merge with target

In [5]:
from datasets import load_dataset

data_load = load_dataset("banking77")

C:\Users\vojta\miniconda3\envs\llm-features\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
load_dataset = pd.DataFrame(data_load['train'])
load_dataset_test = pd.DataFrame(data_load['test'])
load_dataset = pd.concat([load_dataset, load_dataset_test], axis=0)
df['label'] = list(load_dataset['label'])[:len(df)]

In [7]:
# df

In [8]:
df = df.drop(columns=["id", "custom_id"], errors='ignore')
data = pd.get_dummies( 
        df, sparse=False, prefix_sep='_'
    )

In [9]:
# data

In [ ]:
# 1) Libraries
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

# 2) Příprava feature matic X a cílové proměnné y
X = data.drop(columns=["label"])
y = data["label"]
#categorical_columns = X.select_dtypes(include=["object"]).columns
#X = df.drop(columns=categorical_columns)

# 3) Rozdělení na trénovací a testovací sadu
#X_train, X_test, y_train, y_test = train_test_split(X, y,
#                                                    test_size=0.2,
#                                                    random_state=42,
#  stratify=y)  # stratify, pokud je to klasifikace s nerovnoměrnými třídami
# Rozdeleno primo v datasetu
X_train = X[:10003]
y_train = y[:10003]
X_test = X[10003:]
y_test = y[10003:]

# ----------------------------------------------------------------
# 4) Definice modelu RandomForestClassifier
model = GradientBoostingClassifier(random_state=42)

# 5) Nastavení rozsahu parametrů pro RandomizedSearchCV
param_dist = {
    "n_estimators": [50, 100, 200],       # Počet stromů v lese
    "max_depth": [3, 5, 10, None],        # Maximální hloubka stromu
    "min_samples_split": [2, 5, 10],      # Minimální počet vzorků pro split
    "min_samples_leaf": [1, 2, 5],        # Minimální počet vzorků v listu
}

# 6) Konfigurace RandomizedSearchCV (n_iter a cv lze upravit dle potřeby)
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,             # kolik náhodných kombinací parametrů prozkoumat
    cv=5,                  # 5-fold cross-validace
    scoring="accuracy",    # metrika, dle které se bude model porovnávat
    random_state=42,
    n_jobs=-1,             # využití všech CPU jader pro rychlejší výpočet
    verbose=1
)

# 7) Trénink modelu s vyhledáváním nejlepších hyperparametrů
random_search.fit(X_train, y_train)

# 8) Vypsání nejlepších parametrů a skóre
print("Nejlepší parametry:", random_search.best_params_)
print("Nejlepší skóre na trénovací cross-validaci:", random_search.best_score_)

# 9) Ověření na testovací sadě
best_model = random_search.best_estimator_  # získáme nejlepší nalezený model
y_pred = best_model.predict(X_test)

# 10) Vyhodnocení
print("Přesnost na testu:", accuracy_score(y_test, y_pred))
print("Classification report na testu:")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [11]:
grid_rfc = {
 'max_depth': [10, 20, 30, 40, 45, 50],
 'max_features': ['log2', 'sqrt', 30],
 'min_samples_leaf': [30, 40, 50, 60, 70, 90],
 'min_samples_split': [20, 30, 50, 60, 90],
 'n_estimators': [ 100, 200, 400, 1000]}

In [11]:
clf = GradientBoostingClassifier(random_state=42)
clf.fit(X_train, y_train)

GradientBoostingClassifier(random_state=42)

In [12]:
y_test_pred = clf.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.10      0.40      0.16        40
           1       0.90      0.93      0.91        40
           2       0.75      0.68      0.71        40
           3       0.87      0.82      0.85        40
           4       0.54      0.47      0.51        40
           5       0.70      0.57      0.63        40
           6       0.81      0.88      0.84        40
           7       0.45      0.70      0.55        40
           8       0.97      0.80      0.88        40
           9       0.66      0.53      0.58        40
          10       0.60      0.07      0.13        40
          11       0.61      0.35      0.44        40
          12       0.57      0.57      0.57        40
          13       0.63      0.47      0.54        40
          14       0.00      0.00      0.00        40
          15       0.51      0.75      0.61        40
          16       0.48      0.53      0.50        40
          17       0.83    